# Módulo 07 — Estrategias Mixtas

**Objetivos**: Calcular equilibrios mixtos con el principio de indiferencia. Verificar con `nashpy`. Aplicar al penalti con datos reales de la Liga española.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import nashpy as nash
    HAS_NASH = True
except ImportError:
    HAS_NASH = False
    print('nashpy no disponible — usando implementación propia')
print('Entorno listo.')

## 1. El principio de indiferencia

En un equilibrio mixto (p*, q*), cada jugador que mezcla hace **indiferente** al rival entre todas las estrategias puras que el rival usa con probabilidad positiva.

Para un juego 2×2 con matrices A y B:
- J1 mezcla (p, 1−p): condición de indiferencia para J2: B[0,0]·p + B[1,0]·(1−p) = B[0,1]·p + B[1,1]·(1−p)
- J2 mezcla (q, 1−q): condición de indiferencia para J1: A[0,0]·q + A[0,1]·(1−q) = A[1,0]·q + A[1,1]·(1−q)

In [ ]:
def mixed_equilibrium_2x2(A, B):
    """
    Calcula el equilibrio mixto de un juego 2x2 usando el principio de indiferencia.
    Devuelve (p*, q*) o None si no existe equilibrio mixto.
    """
    # p* hace indiferente a J2: B[:,0]·p + B[:,1]·(1-p) uniformes
    # B[0,0]*p + B[1,0]*(1-p) = B[0,1]*p + B[1,1]*(1-p)
    # p*(B[0,0]-B[1,0]-B[0,1]+B[1,1]) = B[1,1]-B[1,0]
    denom_p = B[0,0] - B[1,0] - B[0,1] + B[1,1]
    if abs(denom_p) < 1e-12:
        return None
    p_star = (B[1,1] - B[1,0]) / denom_p

    # q* hace indiferente a J1
    denom_q = A[0,0] - A[0,1] - A[1,0] + A[1,1]
    if abs(denom_q) < 1e-12:
        return None
    q_star = (A[1,1] - A[0,1]) / denom_q

    if 0 <= p_star <= 1 and 0 <= q_star <= 1:
        return p_star, q_star
    return None

# Matching Pennies
A_mp = np.array([[1, -1], [-1, 1]])
B_mp = np.array([[-1, 1], [1, -1]])
eq = mixed_equilibrium_2x2(A_mp, B_mp)
print(f'Matching Pennies: p* = {eq[0]:.4f}, q* = {eq[1]:.4f}  (esperado: 0.5, 0.5)')

In [ ]:
# Verificación con nashpy
if HAS_NASH:
    game = nash.Game(A_mp, B_mp)
    for eq_nash in game.support_enumeration():
        print(f'nashpy equilibrio: p={eq_nash[0]}, q={eq_nash[1]}')

## 2. El penalti — datos reales

Walker & Wooders (2001) analizaron penaltis de la Liga española. Los datos estimados:
- Lanzador: prob óptima de tirar a la izquierda ≈ 0.45
- Portero: prob óptima de tirarse a la izquierda ≈ 0.42

Modelamos con tasas de acierto estimadas.

In [ ]:
# Lanzador: Izq (0) o Der (1). Portero: Izq (0) o Der (1)
# A = pagos del lanzador (prob. de gol)
# B = pagos del portero (prob. de parada = 1 - gol)
A_penalty = np.array([[0.60, 0.90],   # Lanzador tira Izq
                       [0.80, 0.55]])  # Lanzador tira Der
B_penalty = 1 - A_penalty

eq_pen = mixed_equilibrium_2x2(A_penalty, B_penalty)
print(f'Equilibrio mixto del penalti:')
print(f'  Lanzador tira Izq con probabilidad: {eq_pen[0]:.3f}')
print(f'  Portero se tira Izq con probabilidad: {eq_pen[1]:.3f}')
u_L = A_penalty[0,0]*eq_pen[1] + A_penalty[0,1]*(1-eq_pen[1])
print(f'  Prob. de gol en equilibrio: {u_L:.3f}')

In [ ]:
# Curvas de pago esperado
ps = np.linspace(0, 1, 200)
ep_izq = A_penalty[0,0]*ps + A_penalty[1,0]*(1-ps)  # portero tira Izq
ep_der = A_penalty[0,1]*ps + A_penalty[1,1]*(1-ps)  # portero tira Der

plt.figure(figsize=(8, 4.5))
plt.plot(ps, ep_izq, '#1a3a5c', lw=2.5, label='Portero: Izquierda')
plt.plot(ps, ep_der, '#b85c00', lw=2.5, label='Portero: Derecha')
plt.axvline(eq_pen[0], color='#8b5cf6', ls='--', lw=1.5, label=f'p* = {eq_pen[0]:.2f}')
plt.scatter([eq_pen[0]], [u_L], color='#8b5cf6', s=80, zorder=5)
plt.xlabel('p — probabilidad del lanzador de tirar a la izquierda')
plt.ylabel('Probabilidad de gol')
plt.title('Equilibrio mixto en el penalti')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Ejercicios

**Ejercicio 1**: Calcula el equilibrio mixto para el juego 'Gallina' (Chicken): A=[[-5,3],[0,1]], B=[[-5,0],[3,1]]. ¿Qué fracción del tiempo cada jugador cede?

**Ejercicio 2**: Prueba distintos valores de A_penalty y observa cómo cambia p*. ¿Qué ocurre si el portero mejora su parada al lado derecho a 0.65 (en lugar de 0.45)?

In [ ]:
# Tu código aquí
